# Ablation Analysis

1. Decompose authored-helpful-notes by rater factor into (a) total authored and (b) fraction CRH. Plus a stacked status barplot.
2. Per-ablation-run summary (vs baseline `coreRatingStatus`). Saved to `ablation_summary.tsv`.
3. Grid plot: % helpful retained by strategy x % dropped.

In [ ]:
data_dir  <- "/home/jnallen/communitynotes/sourcecode/data_pre_june30"
runs_dir  <- "/home/jnallen/orcd/pool/communitynotes_data/ablation_runs/runs"
out_root  <- "/home/jnallen/orcd/pool/communitynotes_data/analysis"
plots_dir <- file.path(out_root, "plots")
dir.create(plots_dir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
library(arrow)
library(data.table)
library(dplyr)
library(tibble)
library(tidyr)
library(ggplot2)
library(patchwork)
library(stringr)
library(purrr)
library(scales)
theme_set(theme_minimal(base_size = 11))

## 1. Rater-factor decomposition

In [ ]:
rater_factors <- read_parquet(file.path(data_dir, "helpfulness_scores.parquet")) |>
  as_tibble() |>
  select(raterParticipantId, coreRaterFactor1) |>
  filter(!is.na(coreRaterFactor1))

scored <- read_parquet(file.path(data_dir, "scored_notes.parquet")) |>
  as_tibble() |>
  select(noteId, coreRatingStatus)

notes_authors <- fread(file.path(data_dir, "notes-00000.tsv"),
                       select = c("noteId", "noteAuthorParticipantId")) |>
  as_tibble()

cat("raters with factor:", nrow(rater_factors), "\n")
cat("scored notes:      ", nrow(scored), "\n")
cat("notes with author: ", nrow(notes_authors), "\n")

In [ ]:
# Per-author counts of authored notes by core scorer status.
per_rater <- notes_authors |>
  inner_join(scored, by = "noteId") |>
  group_by(noteAuthorParticipantId) |>
  summarise(
    n_crh   = sum(coreRatingStatus == "CURRENTLY_RATED_HELPFUL",     na.rm = TRUE),
    n_crnh  = sum(coreRatingStatus == "CURRENTLY_RATED_NOT_HELPFUL", na.rm = TRUE),
    n_nmr   = sum(coreRatingStatus == "NEEDS_MORE_RATINGS",          na.rm = TRUE),
    n_total = n(),
    .groups = "drop"
  ) |>
  rename(raterParticipantId = noteAuthorParticipantId) |>
  inner_join(rater_factors, by = "raterParticipantId") |>
  mutate(frac_crh = n_crh / pmax(n_total, 1))

cat("authors with factor + notes:", nrow(per_rater), "\n")
head(per_rater)

In [ ]:
n_bins <- 15
brks <- seq(min(per_rater$coreRaterFactor1), max(per_rater$coreRaterFactor1),
            length.out = n_bins + 1)

agg <- per_rater |>
  mutate(bin = cut(coreRaterFactor1, breaks = brks, include.lowest = TRUE),
         bin_mid = (brks[as.integer(bin)] + brks[as.integer(bin) + 1]) / 2) |>
  group_by(bin_mid) |>
  summarise(total_notes = sum(n_total),
            total_crh   = sum(n_crh),
            total_crnh  = sum(n_crnh),
            total_nmr   = sum(n_nmr),
            n_raters    = n(),
            .groups = "drop") |>
  mutate(frac_crh = total_crh / total_notes) |>
  arrange(bin_mid)

agg

In [ ]:
bar_w <- diff(brks)[1] * 0.95

p_total <- ggplot(agg, aes(x = bin_mid, y = total_notes)) +
  geom_col(fill = "steelblue", width = bar_w) +
  labs(x = "coreRaterFactor1 (bin midpoint)", y = "total # notes authored",
       title = "Total notes written")

p_frac <- ggplot(agg, aes(x = bin_mid, y = frac_crh)) +
  geom_col(fill = "steelblue", width = bar_w) +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  labs(x = "coreRaterFactor1 (bin midpoint)", y = "fraction CRH",
       title = "Fraction of authored notes rated helpful")

p_decomp <- p_total + p_frac + plot_layout(nrow = 1) +
  plot_annotation(title = "Decomposing authored helpful notes by rater factor")

ggsave(file.path(plots_dir, "rater_factor_decomp.png"), p_decomp,
       width = 12, height = 4, dpi = 200)
ggsave(file.path(plots_dir, "rater_factor_decomp.pdf"), p_decomp,
       width = 12, height = 4)
p_decomp

In [ ]:
status_long <- agg |>
  select(bin_mid, total_crh, total_crnh, total_nmr) |>
  pivot_longer(-bin_mid, names_to = "status", values_to = "n") |>
  mutate(status = recode(status,
                         total_crh  = "Helpful",
                         total_crnh = "Not helpful",
                         total_nmr  = "Needs more ratings"),
         status = factor(status, levels = c("Not helpful", "Needs more ratings", "Helpful")))

p_stack <- ggplot(status_long, aes(x = bin_mid, y = n, fill = status)) +
  geom_col(width = bar_w) +
  scale_fill_manual(values = c("Helpful" = "steelblue",
                                "Needs more ratings" = "grey60",
                                "Not helpful" = "firebrick")) +
  labs(x = "coreRaterFactor1 (bin midpoint)", y = "# notes authored",
       title = "Authored notes by core scorer status", fill = NULL)

ggsave(file.path(plots_dir, "rater_factor_status_stacked.png"), p_stack,
       width = 10, height = 4, dpi = 200)
ggsave(file.path(plots_dir, "rater_factor_status_stacked.pdf"), p_stack,
       width = 10, height = 4)
p_stack

## 2. Per-ablation-run summary

Compares each run's `coreRatingStatus` against the baseline's `coreRatingStatus`. Saves `ablation_summary.tsv`.

In [ ]:
baseline_crh  <- as.character(scored$noteId[scored$coreRatingStatus == "CURRENTLY_RATED_HELPFUL"])
baseline_crnh <- as.character(scored$noteId[scored$coreRatingStatus == "CURRENTLY_RATED_NOT_HELPFUL"])
baseline_nmr  <- as.character(scored$noteId[scored$coreRatingStatus == "NEEDS_MORE_RATINGS"])

cat("baseline CRH: ",  length(baseline_crh),  "\n")
cat("baseline CRNH:",  length(baseline_crnh), "\n")
cat("baseline NMR: ",  length(baseline_nmr),  "\n")

In [ ]:
process_run <- function(rd) {
  name <- basename(rd)
  m <- str_match(name, "^(extreme|central|random)_([0-9.]+)pct_seed([0-9]+)$")
  if (any(is.na(m))) return(NULL)
  sn_path <- file.path(rd, "scored_notes.tsv")
  if (!file.exists(sn_path)) return(NULL)
  sn <- fread(sn_path, select = c("noteId", "coreRatingStatus"))
  run_crh  <- as.character(sn[coreRatingStatus == "CURRENTLY_RATED_HELPFUL",     noteId])
  run_crnh <- as.character(sn[coreRatingStatus == "CURRENTLY_RATED_NOT_HELPFUL", noteId])
  n_nmr    <- sum(sn$coreRatingStatus == "NEEDS_MORE_RATINGS", na.rm = TRUE)
  tibble(
    run                = name,
    strategy           = m[, 2],
    pct                = as.numeric(m[, 3]),
    seed               = as.integer(m[, 4]),
    n_crh              = length(run_crh),
    n_crnh             = length(run_crnh),
    n_nmr              = n_nmr,
    n_new_crh          = length(setdiff(run_crh,  baseline_crh)),
    n_dropped_crh      = length(setdiff(baseline_crh,  run_crh)),
    n_new_crnh         = length(setdiff(run_crnh, baseline_crnh)),
    n_dropped_crnh     = length(setdiff(baseline_crnh, run_crnh)),
    pct_crh_recovered  = length(intersect(run_crh,  baseline_crh))  / length(baseline_crh),
    pct_crnh_recovered = length(intersect(run_crnh, baseline_crnh)) / length(baseline_crnh)
  )
}

run_dirs <- list.dirs(runs_dir, recursive = FALSE)
cat("found", length(run_dirs), "run dirs\n")

summary_df <- map_dfr(run_dirs, process_run)
fwrite(summary_df, file.path(out_root, "ablation_summary.tsv"), sep = "\t")
cat("wrote", nrow(summary_df), "rows to", file.path(out_root, "ablation_summary.tsv"), "\n")
summary_df

## 3. Grid plot: % helpful retained by strategy x % dropped

In [ ]:
grid <- summary_df |>
  group_by(strategy, pct) |>
  summarise(mean_retained = mean(pct_crh_recovered),
            se_retained   = sd(pct_crh_recovered) / sqrt(n()),
            n_seeds       = n(),
            .groups = "drop")
grid

p_grid <- ggplot(grid, aes(x = factor(pct), y = mean_retained,
                            color = strategy, group = strategy)) +
  geom_line() +
  geom_point(size = 3) +
  geom_errorbar(aes(ymin = mean_retained - se_retained,
                    ymax = mean_retained + se_retained),
                width = 0.1) +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  labs(x = "% raters dropped",
       y = "% of baseline CRH notes retained",
       color = "strategy",
       title = "Ablation impact on helpful notes (core scorer)") +
  theme(legend.position = "top")

ggsave(file.path(plots_dir, "ablation_grid_retained.png"), p_grid,
       width = 8, height = 5, dpi = 200)
ggsave(file.path(plots_dir, "ablation_grid_retained.pdf"), p_grid,
       width = 8, height = 5)
p_grid